# Phase 19 — API Serving

This notebook prepares the GenAI Data Analyst Copilot
for API-style consumption.

Responsibilities:

1. Load the production copilot orchestrator
2. Validate API dependencies
3. Normalize copilot responses
4. Provide a single API request handler
5. Validate SQL, RAG, and Hybrid requests
6. Return JSON-serializable responses

In [0]:
# ================================================================
# PHASE 19 — LOAD ORCHESTRATOR
# ================================================================

print("=" * 70)
print("PHASE 19 — API SERVING")
print("=" * 70)

print()
print("=" * 70)
print("LOADING COPILOT ORCHESTRATOR")
print("=" * 70)


In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
print()
print("Orchestrator loaded.")

In [0]:
# ================================================================
# ASK COPILOT CHECK
# ================================================================

print("=" * 70)
print("ASK_COPILOT CHECK")
print("=" * 70)

print(
    "ask_copilot loaded:",
    callable(globals().get("ask_copilot"))
)

if not callable(globals().get("ask_copilot")):

    raise RuntimeError(
        "ask_copilot() is not available."
    )

print()
print("ask_copilot(): PASS")

In [0]:
# ================================================================
# API RESPONSE FORMATTER
# ================================================================

def format_copilot_response(response):
    """
    Convert the internal copilot response into
    a consistent API response.
    """

    if response is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Copilot returned None.",
            "execution_time_ms": None
        }

    if not isinstance(response, dict):

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                "Invalid response type: "
                + type(response).__name__
            ),
            "execution_time_ms": None
        }

    return {
        "success": bool(
            response.get(
                "success",
                False
            )
        ),

        "question": response.get(
            "question"
        ),

        "route": response.get(
            "route"
        ),

        "answer": response.get(
            "answer"
        ),

        "sql": response.get(
            "sql"
        ),

        "data": response.get(
            "data"
        ),

        "sources": response.get(
            "sources",
            []
        ),

        "error": response.get(
            "error"
        ),

        "execution_time_ms": response.get(
            "execution_time_ms"
        )
    }


print(
    "format_copilot_response(): PASS"
)

In [0]:
# ================================================================
# API DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("API DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "ask_copilot",
    "format_copilot_response"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:

        failed_dependencies.append(
            function_name
        )

print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "API serving cannot continue. "
        "Missing functions: "
        + ", ".join(
            failed_dependencies
        )
    )

print()
print("API dependency check: PASS")

In [0]:
# ================================================================
# JSON SERIALIZATION HELPER
# ================================================================

def make_json_serializable(value):
    """
    Convert common Spark/Python objects into
    JSON-serializable structures.
    """

    if value is None:
        return None

    # Spark DataFrame
    if hasattr(value, "collect") and hasattr(value, "columns"):

        try:
            return [
                row.asDict(recursive=True)
                for row in value.collect()
            ]

        except Exception:

            return str(value)

    # Dictionary
    if isinstance(value, dict):

        return {
            str(key): make_json_serializable(item)
            for key, item in value.items()
        }

    # List / tuple
    if isinstance(value, (list, tuple)):

        return [
            make_json_serializable(item)
            for item in value
        ]

    # Primitive values
    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool
        )
    ):

        return value

    # Fallback
    return str(value)


print(
    "make_json_serializable(): PASS"
)

In [0]:
# ================================================================
# FINAL API RESPONSE BUILDER
# ================================================================

def build_api_response(response):
    """
    Convert an internal copilot response into
    a production-ready JSON-compatible response.
    """

    formatted = format_copilot_response(
        response
    )

    formatted["data"] = make_json_serializable(
        formatted.get("data")
    )

    formatted["answer"] = make_json_serializable(
        formatted.get("answer")
    )

    formatted["sources"] = make_json_serializable(
        formatted.get("sources", [])
    )

    return formatted


print(
    "build_api_response(): PASS"
)

In [0]:
# ================================================================
# API REQUEST HANDLER
# ================================================================

def api_request(question):
    """
    Production API entry point.

    Input:
        question: user question as string

    Output:
        JSON-compatible dictionary
    """

    if question is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Question is required.",
            "execution_time_ms": None
        }

    question = str(question).strip()

    if not question:

        return {
            "success": False,
            "question": question,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Question cannot be empty.",
            "execution_time_ms": None
        }

    try:

        response = ask_copilot(
            question
        )

        return build_api_response(
            response
        )

    except Exception as e:

        return {
            "success": False,
            "question": question,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                f"{type(e).__name__}: {str(e)}"
            ),
            "execution_time_ms": None
        }


print(
    "api_request(): PASS"
)

In [0]:
# ================================================================
# TEST 1 — SQL API
# ================================================================

print("=" * 70)
print("TEST 1 — SQL API")
print("=" * 70)

sql_api_response = api_request(
    "Which region generated the highest revenue?"
)

print(
    json.dumps(
        sql_api_response,
        indent=2,
        default=str
    )
)

assert sql_api_response["success"] is True
assert sql_api_response["route"] == "sql"

print()
print("SQL API: PASS")

In [0]:
# ================================================================
# TEST 2 — RAG API
# ================================================================

print("=" * 70)
print("TEST 2 — RAG API")
print("=" * 70)

rag_api_response = api_request(
    "What is the discount policy?"
)

print(
    json.dumps(
        rag_api_response,
        indent=2,
        default=str
    )
)

assert rag_api_response["success"] is True
assert rag_api_response["route"] == "rag"

rag_text = str(
    rag_api_response["answer"]
)

assert (
    "dummy answer"
    not in rag_text.lower()
)

print()
print("RAG API: PASS")

In [0]:
# ================================================================
# TEST 3 — HYBRID API
# ================================================================

print("=" * 70)
print("TEST 3 — HYBRID API")
print("=" * 70)

hybrid_api_response = api_request(
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)

print(
    json.dumps(
        hybrid_api_response,
        indent=2,
        default=str
    )
)

assert hybrid_api_response["success"] is True
assert hybrid_api_response["route"] == "hybrid"

print()
print("HYBRID API: PASS")

In [0]:
# ================================================================
# TEST 4 — INVALID REQUEST
# ================================================================

print("=" * 70)
print("TEST 4 — INVALID REQUEST")
print("=" * 70)

invalid_response = api_request(
    ""
)

print(
    json.dumps(
        invalid_response,
        indent=2,
        default=str
    )
)

assert invalid_response["success"] is False
assert invalid_response["error"] is not None

print()
print("INVALID REQUEST HANDLING: PASS")

In [0]:
# ================================================================
# PHASE 19 — FINAL API VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 19 — API SERVING VALIDATION")
print("=" * 70)

api_tests = [
    (
        "SQL",
        sql_api_response
    ),
    (
        "RAG",
        rag_api_response
    ),
    (
        "HYBRID",
        hybrid_api_response
    ),
    (
        "INVALID",
        invalid_response
    )
]

successful_tests = 0

for test_name, result in api_tests:

    if test_name == "INVALID":

        passed = (
            result.get("success") is False
            and result.get("error") is not None
        )

    else:

        passed = (
            result.get("success") is True
            and result.get("route") is not None
            and result.get("error") is None
        )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{test_name}"
    )

    if passed:
        successful_tests += 1

print()
print(
    "Total API tests:",
    len(api_tests)
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    len(api_tests) - successful_tests
)

print()

if successful_tests == len(api_tests):

    print(
        "PHASE 19 STATUS: PASS ✓"
    )

else:

    print(
        "PHASE 19 STATUS: FAIL ✗"
    )

In [0]:
# ================================================================
# CELL 2 — RESPONSE FORMATTER
# ================================================================

def format_copilot_response(response):
    """
    Convert the internal copilot response
    into a stable API response format.
    """

    if response is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Copilot returned None.",
            "execution_time_ms": None
        }

    if not isinstance(response, dict):

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                "Invalid response type: "
                + type(response).__name__
            ),
            "execution_time_ms": None
        }

    return {
        "success": response.get(
            "success",
            False
        ),

        "question": response.get(
            "question"
        ),

        "route": response.get(
            "route"
        ),

        "answer": response.get(
            "answer"
        ),

        "sql": response.get(
            "sql"
        ),

        "data": response.get(
            "data"
        ),

        "sources": response.get(
            "sources",
            []
        ),

        "error": response.get(
            "error"
        ),

        "execution_time_ms": response.get(
            "execution_time_ms"
        )
    }


print(
    "format_copilot_response(): PASS"
)

In [0]:
# ================================================================
# CELL 3 — API DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("API DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "ask_copilot",
    "format_copilot_response"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:
        failed_dependencies.append(
            function_name
        )

print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "API serving cannot continue. "
        "Missing functions: "
        + ", ".join(
            failed_dependencies
        )
    )

print()
print("API dependency check: PASS")

In [0]:
# ================================================================
# CELL 4 — API REQUEST HANDLER
# ================================================================

def api_request(question):
    """
    Production API request handler.

    Accepts a user question and returns
    a stable API response.
    """

    if question is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Question cannot be None.",
            "execution_time_ms": None
        }

    if not isinstance(question, str):

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Question must be a string.",
            "execution_time_ms": None
        }

    question = question.strip()

    if not question:

        return {
            "success": False,
            "question": "",
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Question cannot be empty.",
            "execution_time_ms": None
        }

    try:

        response = ask_copilot(
            question
        )

        return format_copilot_response(
            response
        )

    except Exception as e:

        return {
            "success": False,
            "question": question,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                f"{type(e).__name__}: {str(e)}"
            ),
            "execution_time_ms": None
        }


print(
    "api_request(): PASS"
)

In [0]:
# ================================================================
# CELL 5 — API HANDLER VALIDATION
# ================================================================

print("=" * 70)
print("API REQUEST HANDLER VALIDATION")
print("=" * 70)

assert callable(
    api_request
)

print(
    "api_request(): PASS"
)

print(
    "API handler validation: PASS"
)

In [0]:
# ================================================================
# CELL 6 — TEST SQL API
# ================================================================

print("=" * 70)
print("TEST — SQL API REQUEST")
print("=" * 70)

sql_api_response = api_request(
    "Which region generated the highest revenue?"
)

print(
    json.dumps(
        sql_api_response,
        indent=2,
        default=str
    )
)

assert sql_api_response["success"] is True
assert sql_api_response["route"] == "sql"
assert sql_api_response["error"] is None

print()
print("SQL API: PASS")

In [0]:
# ================================================================
# CELL 7 — TEST RAG API
# ================================================================

print("=" * 70)
print("TEST — RAG API REQUEST")
print("=" * 70)

rag_api_response = api_request(
    "What is the discount policy?"
)

print(
    json.dumps(
        rag_api_response,
        indent=2,
        default=str
    )
)

assert rag_api_response["success"] is True
assert rag_api_response["route"] == "rag"
assert rag_api_response["error"] is None

rag_answer = str(
    rag_api_response["answer"]
)

assert (
    "dummy answer"
    not in rag_answer.lower()
)

print()
print("RAG API: PASS")

In [0]:
# ================================================================
# CELL 8 — TEST HYBRID API
# ================================================================

print("=" * 70)
print("TEST — HYBRID API REQUEST")
print("=" * 70)

hybrid_api_response = api_request(
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)

print(
    json.dumps(
        hybrid_api_response,
        indent=2,
        default=str
    )
)

assert hybrid_api_response["success"] is True
assert hybrid_api_response["route"] == "hybrid"
assert hybrid_api_response["error"] is None

print()
print("HYBRID API: PASS")

In [0]:
# ================================================================
# CELL 9 — INVALID REQUEST VALIDATION
# ================================================================

print("=" * 70)
print("INVALID REQUEST VALIDATION")
print("=" * 70)

invalid_requests = [
    None,
    "",
    "   ",
    123
]

for request in invalid_requests:

    result = api_request(
        request
    )

    print(
        "Input:",
        repr(request)
    )

    print(
        "Success:",
        result["success"]
    )

    print(
        "Error:",
        result["error"]
    )

    assert result["success"] is False
    assert result["error"] is not None

    print("-" * 70)

print(
    "Invalid request handling: PASS"
)

In [0]:
# ================================================================
# CELL 10 — PHASE 19 FINAL VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 19 — API SERVING VALIDATION")
print("=" * 70)

api_tests = [
    (
        "SQL",
        sql_api_response
    ),
    (
        "RAG",
        rag_api_response
    ),
    (
        "HYBRID",
        hybrid_api_response
    )
]

successful_tests = 0

for route_name, result in api_tests:

    passed = (
        result.get("success") is True
        and result.get("route") is not None
        and result.get("error") is None
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{route_name}"
    )

    if passed:
        successful_tests += 1

print()
print(
    "Total API tests:",
    len(api_tests)
)

print(
    "Successful API tests:",
    successful_tests
)

print(
    "Failed API tests:",
    len(api_tests) - successful_tests
)

if successful_tests == len(api_tests):

    print()
    print(
        "PHASE 19 STATUS: PASS ✓"
    )

else:

    print()
    print(
        "PHASE 19 STATUS: FAIL ✗"
    )